# Generate student essays from current AI models (OpenRouter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/generate_modern.ipynb)

The detector has trained on current Claude and Gemini essays only, and essays from other current
models (for example Argon) can still slip through. This notebook asks one current model per vendor
family, through one [OpenRouter](https://openrouter.ai) key, to write `N_PER_MODEL` student
essays each: the 15 PERSUADE assignments as grade 6-12 students (so the topics match the real
student essays) plus college essays, with the same prompt plan as the Claude essays, including the
"make it sound like a real student wrote it" requests.

The essays go to `My Drive/skolfie/data/raw/fresh_v3/<model>.jsonl`; `retrain_modern.ipynb` adds
them to the repo's Claude and Gemini essays.

**Before you start**
1. Create a key at openrouter.ai (Keys) and add credit (about 10-30 USD covers the default plan;
   step 3 prints an estimate first and stops above `MAX_TOTAL_USD`).
2. In Colab: the key icon on the left (Secrets) -> add `OPENROUTER_API_KEY` with the key and
   turn on notebook access. The key never goes into a file or the repo.
3. **No GPU needed**: a CPU runtime is enough (Runtime -> Change runtime type -> CPU).
4. **Runtime -> Run all**. It is resumable: run it again after a disconnect and it only asks for
   the essays that are missing.

Privacy: only the essay *assignments* are sent to OpenRouter and the model vendors, never a
student's text.

## 1. Settings

In [ ]:
DRIVE = "/content/drive/MyDrive/skolfie"
OUT_DIR = f"{DRIVE}/data/raw/fresh_v3"   # retrain_modern.ipynb reads FRESH_DRIVE_DIR from here

N_PER_MODEL = 500        # essays per model
SEED = 0                 # 0 = the same assignments and personas as the Claude essays
PERSUADE_SHARE = 0.6     # share of PERSUADE assignments; the rest are college essays
CONCURRENCY = 8          # parallel requests per model
MAX_TOTAL_USD = 30.0     # step 3 stops if the estimated cost is higher

# One model per family: the newest OpenRouter model whose id starts with the prefix and
# contains the keyword. Step 3 prints the choice and the alternatives; to pick yourself, put
# exact OpenRouter ids in MODELS (then FAMILIES is ignored).
FAMILIES = {
    "openai":   ("openai/", "gpt"),
    "google":   ("google/", "gemini"),
    "meta":     ("meta-llama/", "llama"),
    "mistral":  ("mistralai/", "mistral"),
    "xai":      ("x-ai/", "grok"),
    "deepseek": ("deepseek/", "deepseek"),
    "qwen":     ("qwen/", "qwen"),
    "moonshot": ("moonshotai/", "kimi"),
}
MODELS = []              # e.g. ["openai/<model>", "mistralai/<model>"]; [] = pick per family
EXTRA_KEYWORDS = ["argon"]  # also add the newest model whose id contains each of these
SKIP_IDS_CONTAINING = [":free", "vision", "guard", "embed", "image", "audio", "tts",
                       "search", "online", "coder", "-math"]

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

## 2. Mount Drive, clone, install, read the key

In [ ]:
import os
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)

def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def git(*args, check=True):
    """Run git; a GITHUB_TOKEN secret reaches it through a credential helper that reads it from
    the environment, so the token never appears in a URL, a command line, an error message or
    .git/config."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    auth = []
    token = secret("GITHUB_TOKEN")
    if token:
        env["SKOLFIE_GITHUB_TOKEN"] = token
        auth = ["-c", "credential.helper=", "-c",
                "credential.helper=!f() { test \"$1\" = get && echo username=x-access-token "
                "&& echo \"password=$SKOLFIE_GITHUB_TOKEN\"; }; f"]
    return subprocess.run(["git", *auth, *args], env=env, check=check)


url = f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    # Clones made by older versions of this notebook kept the token in the remote URL.
    git("-C", "/content/Skolfie", "remote", "set-url", "origin", url)
    git("-C", "/content/Skolfie", "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie")

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e .

key = secret("OPENROUTER_API_KEY")
assert key, ("Add the secret OPENROUTER_API_KEY (key icon on the left) and allow this "
             "notebook to read it.")
os.environ["OPENROUTER_API_KEY"] = key   # only in this runtime's memory
del key
print("Key found.")

## 3. Pick the models and estimate the cost

Reads OpenRouter's public model list (names, release dates, prices). The estimate assumes about
600 output tokens per essay plus up to 1,500 thinking tokens for reasoning models; the real cost
is usually lower. Nothing is generated in this step.

In [ ]:
import datetime as dt

import pandas as pd
import requests

from skolfie.data.generation.student import student_prompt_plan

catalog = requests.get("https://openrouter.ai/api/v1/models", timeout=60).json()["data"]


def usable(m):
    mid = m["id"].lower()
    outs = (m.get("architecture") or {}).get("output_modalities") or ["text"]
    return "text" in outs and not any(s in mid for s in SKIP_IDS_CONTAINING)


def price(m, kind):
    try:
        return float((m.get("pricing") or {}).get(kind) or 0)
    except ValueError:
        return 0.0


by_id = {m["id"]: m for m in catalog}
rows, chosen = [], []
if MODELS:
    unknown = [mid for mid in MODELS if mid not in by_id]
    assert not unknown, f"Not on OpenRouter: {unknown}"
    chosen = list(MODELS)
else:
    for family, (prefix, keyword) in FAMILIES.items():
        cands = sorted((m for m in catalog if m["id"].startswith(prefix)
                        and keyword in m["id"].lower() and usable(m)),
                       key=lambda m: m.get("created") or 0, reverse=True)
        if not cands:
            print(f"{family}: no model found")
            continue
        chosen.append(cands[0]["id"])
        print(f"{family:9s} -> {cands[0]['id']}   (also: "
              + ", ".join(m["id"] for m in cands[1:5]) + ")")
for kw in EXTRA_KEYWORDS:
    cands = sorted((m for m in catalog if kw in m["id"].lower() and usable(m)),
                   key=lambda m: m.get("created") or 0, reverse=True)
    if cands and cands[0]["id"] not in chosen:
        chosen.append(cands[0]["id"])
        print(f"extra '{kw}' -> {cands[0]['id']}")
    elif not cands:
        print(f"extra '{kw}': no model found on OpenRouter")

specs = student_prompt_plan(N_PER_MODEL, seed=SEED, persuade_share=PERSUADE_SHARE)
tokens_in = sum(len(s.text) // 4 + 20 for s in specs)
tokens_out = sum(int(s.target_words * 1.35) + 50 for s in specs)
total = 0.0
for mid in chosen:
    m = by_id[mid]
    reasoning = "reasoning" in (m.get("supported_parameters") or [])
    out = tokens_out + (1500 * len(specs) if reasoning else 0)
    cost = tokens_in * price(m, "prompt") + out * price(m, "completion")
    total += cost
    created = dt.datetime.fromtimestamp(m.get("created") or 0, dt.timezone.utc).date()
    rows.append({"model": mid, "released": created, "reasoning": reasoning,
                 "$ / M in": round(price(m, "prompt") * 1e6, 2),
                 "$ / M out": round(price(m, "completion") * 1e6, 2),
                 "est. $": round(cost, 2)})
display(pd.DataFrame(rows))
print(f"{len(chosen)} models x {N_PER_MODEL} essays; estimated total ~${total:.2f} "
      f"(limit MAX_TOTAL_USD = ${MAX_TOTAL_USD:.2f})")
assert chosen, "No models chosen."
assert total <= MAX_TOTAL_USD, ("Estimated cost above MAX_TOTAL_USD: lower N_PER_MODEL, "
                                "drop expensive models from MODELS, or raise the limit.")

## 4. Generate

One model at a time; each essay is appended to `OUT_DIR/<model>.jsonl` as soon as it arrives.
Markdown marks (headings, **bold**) are removed, as when a student pastes the answer into a
document. Refusals, cut-off answers and texts under 30 words are dropped (and asked again on a rerun). A
model that fails with a bad id or no credit stops only that model.

In [ ]:
def run(cmd):
    """Run a command and stream its output into the notebook; return its exit code."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
    return proc.wait()


failed = {}
for mid in chosen:
    print(f"\n===== {mid} =====", flush=True)
    code = run([sys.executable, "-u", "scripts/generate_ai_texts.py", "--provider", "openrouter",
                "--model", mid, "--plan", "student", "--n", str(N_PER_MODEL),
                "--seed", str(SEED), "--persuade-share", str(PERSUADE_SHARE),
                "--concurrency", str(CONCURRENCY), "--plain-text", "--out-dir", OUT_DIR])
    if code != 0:
        failed[mid] = code
print("\nFailed models:", failed or "none")

## 5. Check what came back

In [ ]:
import glob
import json

rows = []
for path in sorted(glob.glob(f"{OUT_DIR}/*.jsonl")):
    recs = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            try:
                recs.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    if not recs:
        continue
    words = pd.Series([len(r["text"].split()) for r in recs])
    rows.append({"generator": recs[0]["generator"], "essays": len(recs),
                 "median words": int(words.median()), "min": int(words.min()),
                 "max": int(words.max()),
                 "markdown-ish": sum(r["text"].lstrip().startswith(("#", "**")) for r in recs)})
summary = pd.DataFrame(rows)
display(summary)
if rows:
    sample = recs[0]
    print(f"\nExample ({sample['generator']}):\n")
    print(sample["text"][:1200])

## Next

Tell Claude in the project thread that this has finished; it reads the files from Drive, checks
them, and sets up the held-out model for the retrain (one new family is kept out of training so
the test shows how the detector does on a model it has never seen). Then run
`retrain_modern.ipynb`.